# **Regression**

## Objectives

*   Fit and evaluate a regression model to predict SalePrice


## Inputs

* outputs/datasets/collection/house_prices_records.csv
* Instructions on which variables to use for data cleaning and feature engineering. They are found in their respective notebooks.

## Outputs

* Train set (features and target)
* Test set (features and target)
* ML pipeline to predict SalePrice
* labels map
* Feature Importance Plot



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\farib\\Projects\\Heritage-House-Price-Insight-Predictor\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\farib\\Projects\\Heritage-House-Price-Insight-Predictor'

# Load Data

In [4]:
import numpy as np
import pandas as pd
df = (pd.read_csv("outputs/datasets/collection/house_prices_records.csv")
      )

print(df.shape)
df.head(3)


(1460, 24)


,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,NaN,460,RFn,...,80.0,0.0,0,8,6,1262,NaN,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,NaN,2001,2002,223500


In [6]:
df_modeling = df.copy()
df_modeling.head(3)

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,NaN,460,RFn,...,80.0,0.0,0,8,6,1262,NaN,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,NaN,2001,2002,223500


We add the binary flags (HasOpenPorch, HasMasonry) before the pipeline.
Pipelines do not support dynamic column creation inside by default — so we do this outside and feed into TrainSet.

In [7]:
# Create binary indicator columns
df_modeling['HasOpenPorch'] = (df_modeling['OpenPorchSF'] > 0).astype(int)
df_modeling['HasMasonry'] = (df_modeling['MasVnrArea'] > 0).astype(int)


---

# MP Pipeline: Regressor

## Create ML pipeline

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from feature_engine.imputation import MeanMedianImputer, CategoricalImputer, ArbitraryNumberImputer
from feature_engine.selection import DropFeatures
from feature_engine.encoding import OrdinalEncoder
from feature_engine.transformation import (
    PowerTransformer, BoxCoxTransformer, YeoJohnsonTransformer, LogTransformer
)
# ML algorithms
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor


def PipelineOptimization(model):
    # MODELING PIPELINE
    modeling_pipeline = Pipeline([
    
        # --- 1. Data Cleaning ---
        ("drop_features", DropFeatures(features_to_drop=["EnclosedPorch", "WoodDeckSF"])),
        ("median_imputer", MeanMedianImputer(imputation_method="median", variables=[
            "2ndFlrSF", "BedroomAbvGr", "LotFrontage"
        ])),
        ("constant_imputer", ArbitraryNumberImputer(arbitrary_number=-1, variables=["GarageYrBlt"])),
        ("mode_imputer", CategoricalImputer(imputation_method="frequent", variables=[
            "BsmtExposure", "BsmtFinType1", "GarageFinish"
        ])),

        # --- 2. Feature Engineering ---
        # 2.1 Drop highly sparse features (already transformed into binary outside)
        ("drop_sparse", DropFeatures(features_to_drop=["OpenPorchSF", "MasVnrArea"])),

        # 2.2 Ordinal Encoding
        ("ordinal_encoder", OrdinalEncoder(encoding_method="arbitrary", variables=[
            "BsmtExposure", "BsmtFinType1", "GarageFinish", "KitchenQual"
        ])),

        # 2.3 Numerical Transformation
        ("power_1stFlrSF", PowerTransformer(variables=["1stFlrSF"])),
        ("yeojohnson_group1", YeoJohnsonTransformer(variables=[
            "BedroomAbvGr", "BsmtFinSF1", "BsmtUnfSF", "GarageArea", "YearBuilt", "YearRemodAdd",
            "GrLivArea", "LotArea", "LotFrontage", "OverallCond", "TotalBsmtSF"
        ])),
        ("log_garageyrblt", LogTransformer(variables=["GarageYrBlt"], base="10")),
        ("boxcox_overallqual", BoxCoxTransformer(variables=["OverallQual"])),

        # 2.4 Drop correlated features (manual protection of important features)
        ("drop_correlated", DropFeatures(features_to_drop=["1stFlrSF", "GarageYrBlt"])),

        # --- 3. Feature Scaling ---
        ("feat_scaling", StandardScaler()),

        # --- 4. Feature Selection (e.g., LassoCV, RandomForest etc.) ---
        ("feat_selection", SelectFromModel(estimator=model)),

        # --- 5. Final Model ---
        ("model", model)
    ])

    return modeling_pipeline


Custom Class for hyperparameter optimisation

In [9]:
from sklearn.model_selection import GridSearchCV


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")
            model = PipelineOptimization(self.models[key])

            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring)
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)

        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]

        return df[columns], self.grid_searches


---

## Split Train Test Set

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['SalePrice'], axis=1),
    df['SalePrice'],
    test_size=0.2,
    random_state=0
)

print("* Train set:", X_train.shape, y_train.shape,
      "\n* Test set:",  X_test.shape, y_test.shape)


* Train set: (1168, 23) (1168,) 
* Test set: (292, 23) (292,)


---

# Push files to Repo

* If you do not need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.